<a href="https://colab.research.google.com/github/semanrbingl/ai-ecommerce-product-generator/blob/main/notebooks/05_llm_description.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import torch

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Kullanılan cihaz:", device)

Kullanılan cihaz: cuda


In [3]:
from google.colab import userdata
from openai import OpenAI

api_key = userdata.get('OPEN_API_KEY')
client = OpenAI(api_key=api_key)

response = client.chat.completions.create(
    model="gpt-4o-mini",
    messages=[{"role": "user", "content": "Merhaba, bu bir bağlantı testi. Tek kelimeyle cevap ver: Tamam"}]
)

print(response.choices[0].message.content)

Tamam


In [4]:
import kagglehub
path = kagglehub.dataset_download("paramaggarwal/fashion-product-images-small")
DATA_DIR = path
print(DATA_DIR)

Using Colab cache for faster access to the 'fashion-product-images-small' dataset.
/kaggle/input/fashion-product-images-small


In [5]:
import os
import pandas as pd

df = pd.read_csv(os.path.join(DATA_DIR, "styles.csv"), on_bad_lines="skip")
df["image_path"] = df["id"].astype(str).apply(
    lambda x: os.path.join(DATA_DIR, "images", x + ".jpg")
)
df = df[df["image_path"].apply(os.path.exists)].reset_index(drop=True)

TOP_N = 20
MIN_SAMPLES = 500
counts = df["articleType"].value_counts()
raw_counts = counts.head(TOP_N)
top_classes = raw_counts[raw_counts >= MIN_SAMPLES].index.tolist()

df_model = df[df["articleType"].isin(top_classes)].reset_index(drop=True).copy()

class_names = sorted(df_model["articleType"].unique())
class_to_idx = {name: i for i, name in enumerate(class_names)}
idx_to_class = {i: name for name, i in class_to_idx.items()}

df_model["label"] = df_model["articleType"].map(class_to_idx)

print("Sınıf sayısı:", len(class_names))
print("Toplam görsel:", len(df_model))

Sınıf sayısı: 20
Toplam görsel: 33142


In [6]:
from google.colab import drive
drive.mount('/content/drive')

from torchvision import models
import torch.nn as nn

SAVE_DIR = "/content/drive/MyDrive/ai_ecommerce_product_generator"
checkpoint_path = os.path.join(SAVE_DIR, "baseline_resnet50.pth")
checkpoint = torch.load(checkpoint_path, map_location=device)

num_classes = len(checkpoint["class_to_idx"])
model = models.resnet50(weights=None)
model.fc = nn.Linear(model.fc.in_features, num_classes)
model.load_state_dict(checkpoint["model_state_dict"])
model = model.to(device)
model.eval()

print("Model yüklendi, sınıf sayısı:", num_classes)

Mounted at /content/drive
Model yüklendi, sınıf sayısı: 20


In [7]:
from torchvision import transforms
from PIL import Image

IMG_SIZE = 224
imagenet_mean = [0.485, 0.456, 0.406]
imagenet_std = [0.229, 0.224, 0.225]

eval_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(mean=imagenet_mean, std=imagenet_std),
])

def get_product_info(row):
    image = Image.open(row["image_path"]).convert("RGB")
    input_tensor = eval_transform(image).unsqueeze(0).to(device)

    with torch.no_grad():
        output = model(input_tensor)
        probs = torch.softmax(output, dim=1)
        confidence, pred_idx = torch.max(probs, 1)

    predicted_category = idx_to_class[pred_idx.item()]

    info = {
        "predicted_category": predicted_category,
        "confidence": round(confidence.item(), 2),
        "gender": row.get("gender"),
        "color": row.get("baseColour"),
        "season": row.get("season"),
        "usage": row.get("usage"),
    }

    info = {k: v for k, v in info.items() if pd.notna(v)}
    return info

In [8]:
sample_row = df_model.sample(1, random_state=7).iloc[0]
info = get_product_info(sample_row)
print(info)

{'predicted_category': 'Sunglasses', 'confidence': 0.97, 'gender': 'Men', 'color': 'Black', 'season': 'Winter', 'usage': 'Casual'}


In [9]:
def build_prompt(info):
  details = []
  if "predicted_category" in info:
    details.append(f"Kategori: {info['predicted_category']}")
  if "color" in info:
    details.append(f"Renk: {info['color']}")
  if "gender" in info:
    details.append(f"Hedef kitle: {info['gender']}")
  if "season" in info:
    details.append(f"Sezon: {info['season']}")
  if "usage" in info:
    details.append(f"Kullanım amacı: {info['usage']}")

  details_text = "\n".join(details)

  prompt = f"""Aşağıdaki bilgilere sahip bir e-ticaret ürünü için kısa bir ürün açıklaması yaz.

Ürün bilgileri:
{details_text}

Kurallar:
- Sadece yukarıda verilen bilgileri kullan, ekstra özellik uydurma (malzeme, marka, fiyat gibi bilgiler ekleme).
- 2-3 cümle uzunluğunda olsun.
- Satış diline uygun, akıcı bir Türkçe kullan.
- Confidence/tahmin gibi teknik terimlerden bahsetme, sadece ürünün kendisinden bahset.
"""
  return prompt

print(build_prompt(info))


Aşağıdaki bilgilere sahip bir e-ticaret ürünü için kısa bir ürün açıklaması yaz.
 
Ürün bilgileri:
Kategori: Sunglasses
Renk: Black
Hedef kitle: Men
Sezon: Winter
Kullanım amacı: Casual

Kurallar:
- Sadece yukarıda verilen bilgileri kullan, ekstra özellik uydurma (malzeme, marka, fiyat gibi bilgiler ekleme).
- 2-3 cümle uzunluğunda olsun.
- Satış diline uygun, akıcı bir Türkçe kullan.
- Confidence/tahmin gibi teknik terimlerden bahsetme, sadece ürünün kendisinden bahset.



In [10]:
def generate_description(info):
  prompt = build_prompt(info)
  response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[{"role": "user", "content": prompt}],
        temperature=0.7,
    )
  return response.choices[0].message.content

description = generate_description(info)
print(description)

Kış sezonunda şıklığınızı tamamlayacak bu siyah güneş gözlüğü, gündelik kullanım için ideal bir seçenektir. Erkekler için tasarlanmış modern ve şık görünümüyle stilinize zarif bir dokunuş katın. Casual kombinlerinizi tamamlamak için mükemmel bir aksesuar!


In [11]:
def full_pipeline(row):
  info = get_product_info(row)
  description = generate_description(info)
  return info, description

samples = df_model.sample(3, random_state=21)

for _, row in samples.iterrows():
  info, description = full_pipeline(row)
  print("Görsel:", row["image_path"])
  print("Bilgiler:", info)
  print("Açıklama:", description)
  print("-" * 50)

Görsel: /kaggle/input/fashion-product-images-small/images/54800.jpg
Bilgiler: {'predicted_category': 'Jeans', 'confidence': 0.96, 'gender': 'Men', 'color': 'Blue', 'season': 'Summer', 'usage': 'Casual'}
Açıklama: Yaz sezonu için ideal bir seçim olan mavi jeans, şıklığı ve rahatlığı bir arada sunuyor. Günlük kullanımda rahatlıkla kombinleyebileceğiniz bu parçayla, stilinizi tamamlayarak her ortamda kendinizi özgür hissetmenizi sağlayın. Casual tarzınızı yansıtan bu jeans ile yaz aylarının tadını çıkarın!
--------------------------------------------------
Görsel: /kaggle/input/fashion-product-images-small/images/27202.jpg
Bilgiler: {'predicted_category': 'Shirts', 'confidence': 0.75, 'gender': 'Men', 'color': 'Red', 'season': 'Summer', 'usage': 'Casual'}
Açıklama: Yaz sezonu için ideal bir seçim olan kırmızı tişört, günlük kombinlerinizi şıklıkla tamamlayacak. Rahat ve casual tasarımıyla erkekler için özel olarak hazırlanmış bu tişört, hem konforlu hem de göz alıcı bir görünüm sunuyor. S

In [12]:
category_tr = {
    "Tshirts": "tişört", "Shirts": "gömlek", "Casual Shoes": "günlük ayakkabı",
    "Sports Shoes": "spor ayakkabı", "Formal Shoes": "klasik ayakkabı",
    "Watches": "saat", "Handbags": "el çantası", "Heels": "topuklu ayakkabı",
    "Jeans": "kot pantolon", "Kurtas": "kurta", "Perfume and Body Mist": "parfüm",
    "Sandals": "sandalet", "Socks": "çorap", "Sunglasses": "güneş gözlüğü",
    "Tops": "üst giyim", "Wallets": "cüzdan", "Backpacks": "sırt çantası",
    "Belts": "kemer", "Briefs": "boxer", "Flip Flops": "terlik",
}

In [15]:
def get_product_info(row):
    image = Image.open(row["image_path"]).convert("RGB")
    input_tensor = eval_transform(image).unsqueeze(0).to(device)

    with torch.no_grad():
        output = model(input_tensor)
        probs = torch.softmax(output, dim=1)
        confidence, pred_idx = torch.max(probs, 1)

    predicted_category_en = idx_to_class[pred_idx.item()]
    predicted_category_tr = category_tr.get(predicted_category_en, predicted_category_en)

    info = {
        "predicted_category": predicted_category_tr,
        "confidence": round(confidence.item(), 2),
        "gender": row.get("gender"),
        "color": row.get("baseColour"),
        "season": row.get("season"),
        "usage": row.get("usage"),
    }

    info = {k: v for k, v in info.items() if pd.notna(v)}
    return info

In [16]:
samples = df_model.sample(3, random_state=21)

for _, row in samples.iterrows():
    info, description = full_pipeline(row)
    print("Görsel:", row["image_path"])
    print("Bilgiler:", info)
    print("Açıklama:", description)
    print("-" * 50)

Görsel: /kaggle/input/fashion-product-images-small/images/54800.jpg
Bilgiler: {'predicted_category': 'kot pantolon', 'confidence': 0.96, 'gender': 'Men', 'color': 'Blue', 'season': 'Summer', 'usage': 'Casual'}
Açıklama: Yaz sezonu için mükemmel bir tercih olan mavi kot pantolon, günlük kullanımda rahatlık ve şıklığı bir arada sunuyor. Her erkeğin gardırobunda bulunması gereken bu casual parça, farklı kombinlerle stilinize zarif bir dokunuş katacak.
--------------------------------------------------
Görsel: /kaggle/input/fashion-product-images-small/images/27202.jpg
Bilgiler: {'predicted_category': 'gömlek', 'confidence': 0.75, 'gender': 'Men', 'color': 'Red', 'season': 'Summer', 'usage': 'Casual'}
Açıklama: Yaz sezonu için mükemmel bir tercih olan kırmızı gömlek, her erkeğin gardırobunda bulunması gereken casual bir parçadır. Şıklığı ve rahatlığı bir arada sunan bu gömlek ile günlük kombinlerinize enerjik bir dokunuş katın. Sıcak havalarda ferah bir kullanım sağlar, stilinizi tamamlar.

In [17]:
low_conf_samples = []

for _, row in df_model.sample(50, random_state=99).iterrows():
    info = get_product_info(row)
    if info["confidence"] < 0.6:
        low_conf_samples.append((row, info))
    if len(low_conf_samples) >= 1:
        break

row, info = low_conf_samples[0]
description = generate_description(info)

print("Görsel:", row["image_path"])
print("Bilgiler:", info)
print("Açıklama:", description)

Görsel: /kaggle/input/fashion-product-images-small/images/26950.jpg
Bilgiler: {'predicted_category': 'tişört', 'confidence': 0.44, 'gender': 'Women', 'color': 'Maroon', 'season': 'Summer', 'usage': 'Casual'}
Açıklama: Bu maroon tişört, yaz sezonunun vazgeçilmez parçası olarak tasarlandı. Rahat kesimi ve şık rengiyle, günlük kombinlerinizi tamamlayarak hem şıklığınızı hem de konforunuzu artırır. Casual stilinize zarif bir dokunuş katmak için ideal bir seçim!


In [20]:
def build_prompt(info):
    details = []
    if "predicted_category" in info:
        details.append(f"Kategori: {info['predicted_category']}")
    if "color" in info:
        details.append(f"Renk: {info['color']}")
    if "gender" in info:
        details.append(f"Hedef kitle: {info['gender']}")
    if "season" in info:
        details.append(f"Sezon: {info['season']}")
    if "usage" in info:
        details.append(f"Kullanım amacı: {info['usage']}")

    details_text = "\n".join(details)

    confidence_note = ""
    if info.get("confidence", 1.0) < 0.6:
        confidence_note = "\n- Kategori tahmini kesin değil, bu yüzden kategori ismini net bir iddia gibi değil, daha genel/esnek bir ifadeyle kullan (örneğin 'bu tarz bir ürün' gibi)."

    prompt = f"""Aşağıdaki bilgilere sahip bir e-ticaret ürünü için kısa bir ürün açıklaması yaz.

Ürün bilgileri:
{details_text}

Kurallar:
- Sadece yukarıda verilen bilgileri kullan, ekstra özellik uydurma (malzeme, marka, fiyat gibi bilgiler ekleme).
- 2-3 cümle uzunluğunda olsun.
- Satış diline uygun, akıcı bir Türkçe kullan.
- Confidence/tahmin gibi teknik terimlerden bahsetme, sadece ürünün kendisinden bahset.{confidence_note}
"""
    return prompt

In [21]:
description = generate_description(info)
print(description)

Maroon rengindeki bu şık tişört, yaz aylarında günlük kullanım için mükemmel bir seçim. Rahat kesimi ile hem stilinizi tamamlayacak hem de konfor sunacak. Kadınlar için tasarlanmış bu tarz bir ürünle yazın enerjisini yansıtın.


ÖZET

Model tahmini + metadata birleştirilip OpenAI (gpt-4o-mini) ile Türkçe açıklama üretildi.

- Kategori isimleri Türkçeye çevrildi (İngilizce kalınca yanlış terim kullanılıyordu).
- Confidence < 0.6 olduğunda prompt'a "temkinli dil kullan" talimatı eklendi.